**Esercizio 2**: Modificare l'implementazione della classi `LinearRegressionGD` sostituendo il criterio di terminazione dell'addestramento facendo in modo che questo termini quando la funzione $J(w)$ smette di diminuire in modo significativo.

In [ ]:
import numpy as np
import pandas as pd
import os

def mae(y, z):
    '''
    Calcola l'errore medio assoluto
    '''
    return np.sum(np.abs(y-z))/y.shape[0]

class LinearRegressionGD(object):
    def __init__(self, eta=0.001, n_iter=1000, tol = 0.001):
        self.eta = eta
        self.n_iter = n_iter
        self.costi = []
        self.tol = tol

    def fit(self, X, y):
        # Standardizzazione delle feature
        # media 0, deviazione standard 1
        self.mean_ = X.mean(axis=0) # media per colonne;   COSTO Theta(nd)
        self.std_ = X.std(axis=0)   # deviazione standard per colonne; COSTO Theta(nd)
        X_std = (X - self.mean_) / self.std_ # COSTO Theta(nd)

        # Inizializzazione pesi
        self.w_ = np.zeros(1 + X.shape[1]) # COSTO Theta(d)

        for c in range(self.n_iter):  # per n_iter volte
            output = self.net_input(X_std)  # COSTO Theta(nd)
            errors = y - output  # COSTO Theta(n)
            self.w_[1:] += self.eta * X_std.T.dot(errors)/X.shape[0] # COSTO Theta(nd)
            self.w_[0] += self.eta * errors.sum()/X.shape[0] # # COSTO Theta(n)
            # divisione per X.shape[0] per maggiore stabilità
            self.costi.append(mae(y, output))
            if len(self.costi) > 2 and  abs(self.costi[-2] - self.costi[-1]) < self.tol:
                break
        print(c) # per verificare condizione di uscita
        return self

    def net_input(self, X):
        return np.dot(X, self.w_[1:]) + self.w_[0]

    def predict(self, X):
        # Standardizza i dati usando media e std del training set
        X_std = (X - self.mean_) / self.std_
        return self.net_input(X_std)
    
    def get_coef(self):
        """
        Restituisce i coefficienti e l'intercetta del modello
        riferiti ai dati originali (non standardizzati).
        """
        # Coefficienti originali
        coef = self.w_[1:] / self.std_
        
        # Intercetta originale
        intercept = self.w_[0] - np.sum(self.w_[1:] * self.mean_ / self.std_)
        
        return intercept, coef

In [6]:
df = pd.read_csv(os.path.join('dataset', 'trail_running.csv'), sep=';')
df.columns = ['Gara', 'Lunghezza', 'D+', 'Tempo']
df['Lunghezza'] = df['Lunghezza']*1000
df['Tempo'] = pd.to_timedelta(df['Tempo']).dt.total_seconds()/60
df.columns = ['Gara', 'Lunghezza', 'D+', 'Minuti']
df = df[df['Lunghezza'] < 100000]

X = df[ ['Lunghezza', 'D+'] ].values 
y = df['Minuti']

linreg = LinearRegressionGD(eta = 0.001, n_iter=5500, tol=0.001)

linreg.fit(X, y)

previsioni = linreg.predict(X)

# oppure usando i intercetta e coefficienti
intercept, coef = linreg.get_coef()
previsioni = intercept + np.dot(X, coef)

# Valori reali che il modello sta cercando di prevedere
valori_reali = y

# Calcolo del Mean Absolute Error
error = mae(valori_reali, previsioni)

print(f"Mean Absolute Error (MAE): {error:.2f}")

4178
Mean Absolute Error (MAE): 14.56


**Esercizio 3**. Si consideri il dataset contenuto nel file `linear_regression.csv`, costituito da `2000` campioni e `11` colonne totali: le prime `10` rappresentano le feature, mentre l'ultima colonna rappresenta la variabile target `y`.

L'obiettivo dell'esercizio è addestrare un modello di regressione lineare utilizzando la classe `LinearRegressionGD`, già implementata a lezione e eventualmente estesa secondo le specifiche dell'esercizio precedente.

Dopo aver caricato il dataset dal file CSV, si richiede di separare correttamente la matrice delle feature `X` dal vettore target `y`.

Successivamente, il dataset deve essere suddiviso in un *training set* e un *test set* e addestrato sul training set.

Una volta completato l'addestramento, si richiede di valutare le prestazioni del modello calcolando il MAE e sul training set e sul test set. Infine si confrontino e valutino i risultati ottenuti.

In [7]:
df = pd.read_csv(os.path.join('dataset', 'linear_regression.csv'), sep=';', header=None)
X = df.iloc[:,:-1].values 
y = df.iloc[:, -1].values

n = X.shape[0]
train_size = 0.7
train_idxs = np.random.choice(n, size = int(n*train_size), replace=False)
test_idxs =  np.setdiff1d(np.arange(n), train_idxs)

X_train, X_test = X[train_idxs], X[test_idxs]
y_train, y_test = y[train_idxs], y[test_idxs]


linreg = LinearRegressionGD(eta = 0.001, n_iter=10000, tol=0.00001)

linreg.fit(X_train, y_train)

previsioni = linreg.predict(X_train)

# oppure usando i intercetta e coefficienti
intercept, coef = linreg.get_coef()
previsioni = intercept + np.dot(X_train, coef)

# Valori reali che il modello sta cercando di prevedere
valori_reali = y_train

# Calcolo del Mean Absolute Error
error = mae(valori_reali, previsioni)

print(f"Mean Absolute Error (MAE) su training set: {error:.2f}")


6520
Mean Absolute Error (MAE) su training set: 8.04


In [8]:
previsioni = linreg.predict(X_test)

# oppure usando i intercetta e coefficienti
intercept, coef = linreg.get_coef()
previsioni = intercept + np.dot(X_test, coef)

# Valori reali che il modello sta cercando di prevedere
valori_reali = y_test

# Calcolo del Mean Absolute Error
error = mae(valori_reali, previsioni)

print(f"Mean Absolute Error (MAE) su test set: {error:.2f}")

Mean Absolute Error (MAE) su test set: 7.95


# Discesa del gradiente stocastica

Nell'algoritmo *discesa del gradiente*, ad ogni iterazione l'aggiornamento dei coefficienti viene eseguito mediante la seguente formula

$$
w \leftarrow w + \Delta w = w -\eta \nabla J(w)
$$

dove

$$
\nabla J(w) = - X^T\times \texttt{errors}.
$$

Il calcolo di $\nabla J(w)$ utilizza l'intero dataset di input ($X$) che deve essere disponibile per intero in memoria. Su dataset molto grandi questa soluzione può essere impraticabile.

**Discesa del Gradiente Stocastica**. In questa versione $\nabla J(w)$ viene calcolato usando una porzione della matrice $X$ ottenuta selezionando casualmente un numero fissato di righe (campioni). La nuova matrice ottenuta, $X_b$, corrispondente a un blocco o batch di campioni, verrà usata per calcolare $\nabla J(w)$ (e quindi i nuovi campioni) come

$$
\nabla J(w) = - X_b^T\times \texttt{errors}.
$$

In questo modo, in memoria principale si tiene soltanto $X_b$ mentre l'intero dataset può essere letto dalla memoria secondaria.  


Operativamente, durante ogni iterazione, le righe di $X$ (i campioni), vengono partizionate in *blocchi* di dimensione fissata, si costruisce $X_b$ a partire da ciascun blocco e si usa per calcolare il gradiente e aggiornare i coefficienti.

Si osservi che, mentre con Discesa del Gradiente tradizionale, ad ogni iterazione ogni coefficiente viene aggiornato solo una volta, con la Stocastica ogni coefficiente viene aggiornato tante volte quanti sono i blocchi della partizione



Se $X_b$ contiene un singolo campione, si parla di **SGD puro**, con aggiornamento dei pesi campione per campione.

## Lavoro da svolgere

Partendo da una classe `LinearRegressionGD` implementata con discesa del gradiente tradizionale, si richiede di:

1. Creare una nuova classe `LinearRegressionSGD`.

2. Modificare il metodo `fit` in modo che:

    - i campioni vengano mescolati ad ogni iterazione (shuffle);

    - il dataset venga suddiviso in blocchi di dimensione fissata;

    - per ciascun blocco, i pesi vengano aggiornati usando il gradiente calcolato solo su quel blocco;

    - a fine iterazione venga calcolata e registrata la funzione di costo media sull’intero dataset, per monitorare la convergenza.

3. Confrontare il comportamento delle due versioni utilizzando dataset sintetici generati dalla funzione `make_regression` della libreria `sklearn.datasets`. 



In [9]:
import matplotlib.pyplot as plt
import numpy as np

class LinearRegressionSGD(object):
    def __init__(self, eta=0.001, n_iter=1000, tol=1e-5, batch_size=None):
        self.eta = eta
        self.n_iter = n_iter
        self.costi_ = []
        self.tol_ = tol
        self.batch_size_ = batch_size

    def fit(self, X, y):
        self.batch_size_ = X.shape[0] if self.batch_size_ == None else self.batch_size_

        # Standardizzazione delle feature
        self.mean_ = X.mean(axis=0)
        self.std_ = X.std(axis=0)

        X_std = (X - self.mean_) / self.std_
        
        # Inizializzazione pesi
        self.w_ = np.zeros(1 + X.shape[1])
        
        for _ in range(self.n_iter):
            # shuffle degli indici
            shuffle_idxs = np.arange(X.shape[0])
            np.random.shuffle(shuffle_idxs)

            for b_idx in range(0, X.shape[0], self.batch_size_):
                X_b = X_std[shuffle_idxs[b_idx:b_idx+self.batch_size_]]
                y_b = y[shuffle_idxs[b_idx:b_idx+self.batch_size_]]

                output = self.net_input(X_b)
                errors = y_b - output
                
                self.w_[1:] += self.eta * X_b.T.dot(errors)/self.batch_size_
                self.w_[0] += self.eta * errors.sum()/self.batch_size_
            
            output = self.net_input(X_std)
            self.costi_.append( mae(y, output) )
            if len(self.costi_) >= 2:
                if np.abs(self.costi_[-1] - self.costi_[-2]) < self.tol_:
                    break
        return self

    def net_input(self, X):
        return np.dot(X, self.w_[1:]) + self.w_[0]
        
    def predict(self, X):
        # Standardizza i dati usando media e std del training set
        X_std = (X - self.mean_) / self.std_
        
        return self.net_input(X_std)
    
    def get_coef(self):
        """
        Restituisce i coefficienti e l'intercetta del modello
        riferiti ai dati originali (non standardizzati).
        """

        # Coefficienti originali
        coef = self.w_[1:] / self.std_
        # Intercetta originale
        intercept = self.w_[0] - np.sum(self.w_[1:] * self.mean_ / self.std_)
        
        return intercept, coef
    
def mae(y, z):
    '''
    Calcola l'errore medio assoluto
    '''
    return np.sum(np.abs(y-z))/y.shape[0]

In [12]:
from sklearn.datasets import make_regression

X, y = make_regression(
    n_samples=2000,       # numero di campioni
    n_features=10,        # numero di feature
    noise=10.0,          # rumore da aggiungere ai target
    random_state=42      # per riproducibilità
)

#linregSGD = LinearRegressionSGD(batch_size=1, n_iter=10000, tol=0.00001)
#linregSGD.fit(X, y)
#errorSGD = mae(y, linregSGD.predict(X))

#print(f"SGD Mean Absolute Error (MAE): {errorSGD:.2f}. Iterazioni: {len(linregSGD.costi_)}")

linregSGD = LinearRegressionSGD(batch_size=500, n_iter=10000, tol=0.00001)
linregSGD.fit(X, y)
errorSGD = mae(y, linregSGD.predict(X))

print(f"SGD Mean Absolute Error (MAE): {errorSGD:.2f}. Iterazioni: {len(linregSGD.costi_)}")

linregGD = LinearRegressionSGD(n_iter=10000, tol=0.00001)
linregGD.fit(X, y)
errorGD = mae(y, linregGD.predict(X))

print(f"GD Mean Absolute Error (MAE): {errorGD:.2f}. Iterazioni: {len(linregGD.costi_)}")



SGD Mean Absolute Error (MAE): 8.00. Iterazioni: 1937
GD Mean Absolute Error (MAE): 8.01. Iterazioni: 6608


**Esercizio**. Modificare la classe `LinearRegressionSGD`introducendo una condizione di arresto basata su *validation set*. Ovvero, durante l'addestramento, l'algoritmo non deve limitarsi a controllare la convergenza sul training set, ma deve monitorare anche l'errore su un insieme di validazione fornito separatamente.

In particolare, ad ogni epoca si richiede di:

- calcolare il valore della funzione di costo sia sul training set sia sul validation set;
- mantenere in memoria il miglior valore di errore sul validation set osservato fino a quel momento
- salvare i parametri del modello (pesi e intercetta) corrispondenti a tale valore minimo.

La procedura di addestramento deve essere interrotta nel caso in cui l'errore sul validation set non migliori per un numero prefissato di epoche consecutive (parametro `patience` da aggiungere all'elenco).

Al termine dell'addestramento, il modello deve:

- ripristinare i parametri corrispondenti al miglior valore di validation error;
- restituire tali parametri come soluzione finale.

Questa modifica consente di implementare una strategia di arresto, utile a prevenire fenomeni di overfitting, in quanto l'arresto dell'algoritmo avviene nel momento in cui le prestazioni su dati non visti (validation set) smettono di migliorare, anche se l'errore sul training set continua a diminuire.